In [1]:
from data_loader import load_mimic_tables                                      
                                                                                 
tables = load_mimic_tables()                                                                                                        
                                                                                 
# See all available tables:                                                    
print(tables.keys())  

/Users/huseyin/Codes/gpt-medic/explore/data_loader.py:26: DtypeWarning: Columns (0: administration_type, 1: barcode_type, 2: reason_for_no_barcode, 3: complete_dose_not_given, 4: dose_due, 5: dose_due_unit, 6: dose_given, 7: dose_given_unit, 8: will_remainder_of_dose_be_given, 9: product_unit, 10: product_code, 11: product_description, 12: product_description_other, 13: infusion_rate_adjustment, 14: infusion_rate_unit, 15: route, 16: infusion_complete, 17: completion_interval, 18: new_iv_bag_hung, 19: continued_infusion_in_other_location, 20: restart_interval, 21: side, 22: site, 23: non_formulary_visual_verification) have mixed types. Specify dtype option on import or set low_memory=False.
  tables[table_name] = pd.read_csv(file, compression="gzip")


dict_keys(['hosp.poe', 'hosp.d_hcpcs', 'hosp.poe_detail', 'hosp.patients', 'hosp.diagnoses_icd', 'hosp.emar_detail', 'hosp.provider', 'hosp.prescriptions', 'hosp.drgcodes', 'hosp.d_icd_diagnoses', 'hosp.d_labitems', 'hosp.transfers', 'hosp.admissions', 'hosp.labevents', 'hosp.pharmacy', 'hosp.procedures_icd', 'hosp.hcpcsevents', 'hosp.services', 'hosp.d_icd_procedures', 'hosp.omr', 'hosp.emar', 'hosp.microbiologyevents', 'icu.datetimeevents', 'icu.caregiver', 'icu.ingredientevents', 'icu.inputevents', 'icu.procedureevents', 'icu.d_items', 'icu.chartevents', 'icu.icustays', 'icu.outputevents'])


In [30]:
import pandas as pd

gsn_atc = pd.read_csv('../data/gsn_atc_ndc_mapping.csv', dtype=str)

prescriptions = tables['hosp.prescriptions'].copy()
prescriptions['gsn'] = prescriptions['gsn'].astype(str).str.zfill(6)

prescriptions = prescriptions.merge(
    gsn_atc[['gsn', 'atc']].drop_duplicates('gsn'),
    on='gsn',
    how='left'
).rename(columns={'atc': 'atc_code'})

print(f"Total rows:  {len(prescriptions)}")
print(f"ATC mapped:  {prescriptions['atc_code'].notna().sum()}")
prescriptions[['drug', 'gsn', 'atc_code']].dropna(subset=['atc_code']).head(10)

Total rows:  18087
ATC mapped:  15563


,drug,gsn,atc_code
9,Insulin,001723,A10AE05
10,Insulin,001723,A10AE05
11,Insulin,001723,A10AE05
12,Insulin,001723,A10AE05
13,Insulin,001723,A10AE05
14,Insulin,001723,A10AE05
15,Insulin,001723,A10AE05
16,Insulin,001723,A10AE05
17,Insulin,001723,A10AE05
18,Insulin,001723,A10AE05


In [49]:
import pandas as pd
from tokenizer.medication import MedicationTokenizer

# GSN → ATC mapping via local file
gsn_atc = pd.read_csv('../data/gsn_atc_ndc_mapping.csv', dtype=str)

prescriptions = tables['hosp.prescriptions'].copy()
prescriptions['gsn'] = prescriptions['gsn'].astype(str).str.zfill(6)

prescriptions = prescriptions.merge(
    gsn_atc[['gsn', 'atc']].drop_duplicates('gsn'),
    on='gsn',
    how='left'
).rename(columns={'atc': 'atc_code'})

print(f"Total rows:  {len(prescriptions)}")
print(f"ATC mapped:  {prescriptions['atc_code'].notna().sum()}")

# Run MedicationTokenizer
med_tokenizer = MedicationTokenizer()
vocab = med_tokenizer.build_vocabulary(prescriptions.dropna(subset=['atc_code']))

print(f"Vocabulary size: {len(vocab)}")
vocab

Total rows:  18087
ATC mapped:  15563
Vocabulary size: 190


{'ATC_4_A': 0,
 'ATC_4_B': 1,
 'ATC_4_C': 2,
 'ATC_4_D': 3,
 'ATC_4_E': 4,
 'ATC_4_F': 5,
 'ATC_4_G': 6,
 'ATC_4_H': 7,
 'ATC_4_M': 8,
 'ATC_4_X': 9,
 'ATC_A01': 10,
 'ATC_A02': 11,
 'ATC_A03': 12,
 'ATC_A04': 13,
 'ATC_A05': 14,
 'ATC_A06': 15,
 'ATC_A07': 16,
 'ATC_A09': 17,
 'ATC_A10': 18,
 'ATC_A11': 19,
 'ATC_A12': 20,
 'ATC_A16': 21,
 'ATC_B01': 22,
 'ATC_B02': 23,
 'ATC_B03': 24,
 'ATC_B05': 25,
 'ATC_C01': 26,
 'ATC_C02': 27,
 'ATC_C03': 28,
 'ATC_C04': 29,
 'ATC_C05': 30,
 'ATC_C07': 31,
 'ATC_C08': 32,
 'ATC_C09': 33,
 'ATC_C10': 34,
 'ATC_D01': 35,
 'ATC_D02': 36,
 'ATC_D03': 37,
 'ATC_D04': 38,
 'ATC_D06': 39,
 'ATC_D07': 40,
 'ATC_D10': 41,
 'ATC_D11': 42,
 'ATC_G02': 43,
 'ATC_G03': 44,
 'ATC_G04': 45,
 'ATC_H01': 46,
 'ATC_H02': 47,
 'ATC_H03': 48,
 'ATC_H04': 49,
 'ATC_H05': 50,
 'ATC_J01': 51,
 'ATC_J02': 52,
 'ATC_J05': 53,
 'ATC_J07': 54,
 'ATC_L01': 55,
 'ATC_L02': 56,
 'ATC_L03': 57,
 'ATC_M01': 58,
 'ATC_M02': 59,
 'ATC_M03': 60,
 'ATC_M04': 61,
 'ATC_M05': 62,
 '

In [48]:
prescriptions[prescriptions['gsn'] == '000nan']['drug'].value_counts()

drug
Sodium Chloride 0.9%  Flush                    585
Bag                                            454
Iso-Osmotic Dextrose                           313
0.9% Sodium Chloride                           310
5% Dextrose                                    160
Sterile Water                                  147
SW                                              77
Soln                                            68
Vial                                            67
Iso-Osmotic Sodium Chloride                     60
NS                                              47
Syringe (0.9% Sodium Chloride)                  46
Syringe                                         40
D5W                                             39
Prismasate (B32 K2)                             18
Dextrose 5%                                     13
0.83% Sodium Chloride                           12
Yellow CADD Cassette                            10
Prismasate (B22 K4)                             10
Bupivacaine 0.1%          

In [45]:
prescriptions["gsn"][prescriptions["gsn"] =='000nan']

0        000nan
1        000nan
2        000nan
3        000nan
4        000nan
          ...  
17582    000nan
17879    000nan
17880    000nan
18063    000nan
18064    000nan
Name: gsn, Length: 2519, dtype: object